# ⚖️ DPO — 선호 학습으로 답변 품질 높이기
"A답이 B답보다 낫다"는 **선호 쌍**으로 학습합니다. RLHF의 쉽고 안정적인 대체재.
> 💡 SFT가 "무엇을 답할지"라면, DPO는 "어떻게 더 잘 답할지"를 가르칩니다.


In [ ]:
%%capture
!pip install unsloth
!pip install --no-deps "xformers<0.0.30" trl peft accelerate bitsandbytes datasets


## 🔑 HuggingFace 로그인 (맨 먼저!)
write 토큰을 붙여넣으세요. 비공개 데이터셋 로드 + 모델 업로드에 필요해요.


In [ ]:
from huggingface_hub import notebook_login
notebook_login()


In [ ]:
from unsloth import FastModel
import torch
model, tokenizer = FastModel.from_pretrained(model_name="unsloth/gemma-4-E2B-it", dtype=None, max_seq_length=1024, load_in_4bit=True, full_finetuning=False)
print("✅ 베이스 모델 로딩")


In [ ]:
model = FastModel.get_peft_model(model, r=16, lora_alpha=32, lora_dropout=0, bias="none", random_state=3407,
    finetune_language_layers=True, finetune_attention_modules=True, finetune_mlp_modules=True, finetune_vision_layers=False)


In [ ]:
from unsloth.chat_templates import get_chat_template
tokenizer = get_chat_template(tokenizer, chat_template="gemma-4")


## 📦 내 선호 데이터 (👍 chosen vs 👎 rejected)
Connect AI에서 답변에 누른 **👍/👎로 만든 내 데이터**입니다.


In [ ]:
from datasets import load_dataset
ds = load_dataset("seoul625/ai-visual-lab-brain", data_files="connect-ai-dpo.jsonl", split="train", token=True)
print("내 선호 쌍:", len(ds)); print(ds[0])


## 🎯 DPO 학습
`beta`(0.1)는 원래 모델에서 얼마나 벗어날지. 작을수록 보수적.


In [ ]:
from trl import DPOTrainer, DPOConfig
trainer = DPOTrainer(model=model, ref_model=None, tokenizer=tokenizer, train_dataset=ds,
    args=DPOConfig(per_device_train_batch_size=1, gradient_accumulation_steps=4, warmup_steps=5,
        max_steps=30, learning_rate=5e-5, beta=0.1, logging_steps=1, optim="adamw_8bit",
        lr_scheduler_type="linear", seed=3407, report_to="none"))
trainer.train()
print("🎉 DPO 학습 완료 — 답변이 chosen 쪽으로 정렬됩니다")


## 💾 GGUF로 저장 → LM Studio
토큰 칸에 HuggingFace **write 토큰**을 붙여넣으세요.


In [ ]:
from huggingface_hub import notebook_login
notebook_login()


In [ ]:
model.push_to_hub_gguf("seoul625/내두뇌-v2", tokenizer, quantization_method="q4_k_m", token=True)
print("✅ huggingface.co/seoul625/내두뇌-v2 → LM Studio에서 다운로드")
